# E3-G2 — Sensibilité du bootstrap temporel à la longueur des blocs

**Objectif :** tester si les conclusions E3-G changent lorsque les blocs temporels passent de **1 à 2, 3 ou 5 jours**, sans réentraîner les modèles.

- Même jeu de test : **13 643 cibles communes par horizon**.
- Même définition des contrastes : `RMSE(A) − RMSE(B)` ; négatif = A meilleur.
- **2 000 réplications** par horizon et longueur de bloc ; bootstrap circulaire de jours UTC entiers, avec tous les dispositifs groupés.
- Même traitement des graines : moyenne des RMSE calculées séparément sur les trois graines.
- Les intervalles à 95 % sont **descriptifs**, non corrigés pour les comparaisons multiples.
- **Attention** : seulement 22 jours UTC ; des blocs de 5 jours contiennent peu d'unités temporelles indépendantes. Une stabilité entre longueurs de blocs renforce la robustesse *interne*, pas la généralisation à d'autres bâtiments.

Le notebook vérifie les alignements et sauvegarde les résultats dans un nouveau dossier ; il ne modifie pas E3-G.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json, platform
ROOT=Path.cwd().resolve()
OUTPUTS=ROOT/'outputs'
OUT=OUTPUTS/'E3_G2_BLOCK_LENGTH_SENSITIVITY'
SOURCES={
 'tabular':OUTPUTS/'E3_E2_multidevice_tabular_105'/'03_common_test_predictions.csv',
 'gcn':OUTPUTS/'E3_F2A_GCN_200epochs'/'03_common_test_predictions_all_seeds.csv',
 'sequence':OUTPUTS/'E3_F2B_LSTM_TCN_200epochs'/'03_common_test_predictions_all_seeds.csv'
}
B=2000
BLOCK_LENGTHS=(1,2,3,5)
RNG_SEED=20261104
SEEDS=(42,123,2026)
HORIZONS=(15,30,60,120)
PAIRS=[
 ('G_operational','G_shuffled'),
 ('TCN','G_operational'),
 ('G_native','Persistence'),
 ('Ridge','G_native'),
 ('Ridge','Persistence')
]
assert all(p.is_file() for p in SOURCES.values()),'Fichiers introuvables : '+str({k:str(v) for k,v in SOURCES.items() if not v.is_file()})
OUT.mkdir(parents=True,exist_ok=True)


In [ ]:
def ingest(path,kind):
    df=pd.read_csv(path)
    if kind=='gcn':df=df.rename(columns={'topology':'model'})
    if kind=='tabular':df['seed']=-1
    required={'uuid','timestamp','horizon_min','model','seed','y_true','y_pred','persistence_pred'}
    assert required.issubset(df.columns),(kind,required-set(df.columns))
    df['timestamp']=pd.to_datetime(df.timestamp,utc=True)
    df['uuid']=df.uuid.astype(str)
    df['model']=df.model.astype(str)
    df['seed']=df.seed.astype(int)
    assert df[['y_true','y_pred','persistence_pred']].notna().all().all()
    assert np.isfinite(df[['y_true','y_pred','persistence_pred']].to_numpy()).all()
    assert not df.duplicated(['horizon_min','model','seed','uuid','timestamp']).any()
    return df[['uuid','timestamp','horizon_min','model','seed','y_true','y_pred','persistence_pred']]

data=pd.concat([ingest(path,k) for k,path in SOURCES.items()],ignore_index=True)
assert not data.duplicated(['horizon_min','model','seed','uuid','timestamp']).any()
print(data.groupby(['model','horizon_min']).agg(seeds=('seed','nunique'),rows=('y_true','size')).to_string())


In [ ]:
# Alignement strict des cibles et de la persistance, pour chaque horizon et chaque modèle/graine.
audit=[]
reference={}
for h in HORIZONS:
    sub=data[data.horizon_min.eq(h)]
    ref=(sub[(sub.model=='Ridge') & (sub.seed==-1)]
         .sort_values(['uuid','timestamp']).reset_index(drop=True))
    assert len(ref)==13643,f'Ridge reference missing for {h}'
    key=ref[['uuid','timestamp']]
    for (model,seed),g in sub.groupby(['model','seed']):
        g=g.sort_values(['uuid','timestamp']).reset_index(drop=True)
        assert len(g)==len(ref),(h,model,seed,len(g))
        assert g[['uuid','timestamp']].equals(key),(h,model,seed,'key mismatch')
        assert np.allclose(g.y_true,ref.y_true,atol=1e-5,rtol=0),(h,model,seed,'target mismatch')
        assert np.allclose(g.persistence_pred,ref.persistence_pred,atol=1e-5,rtol=0),(h,model,seed,'persistence mismatch')
        audit.append({'horizon_min':h,'model':model,'seed':seed,'n':len(g),
                      'keys_aligned':True,'targets_aligned':True,'persistence_aligned':True})
    reference[h]=ref
assert set(data.model.unique())=={'Ridge','RandomForest','XGBoost','LSTM','TCN','G_native','G_operational','G_shuffled'}
assert len(audit)==4*(3+5*3)
pd.DataFrame(audit).to_csv(OUT/'01_alignment_audit.csv',index=False)
print('Alignment checks passed:',len(audit))


## Sensibilité des contrastes

Le point estimé ne dépend pas de la longueur des blocs ; seuls les intervalles de confiance changent. Les conclusions sont classées `A_better`, `B_better` ou `inconclusive` selon que l'intervalle se trouve entièrement sous zéro, au-dessus de zéro, ou traverse zéro. Une conclusion `inconclusive` **n'est pas** une preuve d'équivalence.

L'analyse produit un tableau détaillé (80 contrastes), un tableau des transitions par paire/horizon et un récapitulatif de stabilité.


In [ ]:
def metric(y,p):
    return float(np.sqrt(np.mean(np.square(np.asarray(p)-np.asarray(y)))))

rows=[]
baseline_rows=[]
for h in HORIZONS:
    ref=reference[h]
    dates=ref.timestamp.dt.floor('D')
    days=pd.DatetimeIndex(sorted(dates.unique()))
    D=len(days)
    assert D==22, f'Unexpected number of days: {h} -> {D}'
    day_id=days.get_indexer(dates)
    nday=np.bincount(day_id,minlength=D).astype(float)
    assert nday.sum()==13643
    sqday={}
    for model in sorted(data.model.unique()):
        subset=data[(data.horizon_min==h)&(data.model==model)]
        per_seed=[]
        for seed in sorted(subset.seed.unique()):
            g=subset[subset.seed==seed].sort_values(['uuid','timestamp'])
            e2=(g.y_pred.to_numpy()-ref.y_true.to_numpy())**2
            per_seed.append(np.bincount(day_id,weights=e2,minlength=D))
        sqday[model]=np.stack(per_seed)
    e2p=(ref.persistence_pred.to_numpy()-ref.y_true.to_numpy())**2
    sqday['Persistence']=np.bincount(day_id,weights=e2p,minlength=D)[None,:]
    point={model:float(np.sqrt(sums.sum(axis=1)/nday.sum()).mean())
           for model,sums in sqday.items()}
    for model,v in point.items():
        baseline_rows.append({'horizon_min':h,'model':model,'RMSE_mean':v})
    for L in BLOCK_LENGTHS:
        # Independent reproducible generator for each horizon/block length.
        rng=np.random.default_rng(RNG_SEED + 1000*h + L)
        nblocks=int(np.ceil(D/L))
        starts=rng.integers(0,D,size=(B,nblocks))
        picks=((starts[:,:,None]+np.arange(L))%D).reshape(B,-1)[:,:D]
        counts=np.zeros((B,D),dtype=np.int32)
        np.add.at(counts,(np.repeat(np.arange(B),D),picks.reshape(-1)),1)
        denominator=counts@nday
        assert (denominator>0).all()
        boot={}
        for model,sums in sqday.items():
            boot[model]=np.sqrt((sums@counts.T)/denominator[None,:]).mean(axis=0)
        for a,b in PAIRS:
            diff=boot[a]-boot[b]
            lo,hi=np.quantile(diff,[0.025,0.975])
            observed=point[a]-point[b]
            status='A_better' if hi<0 else ('B_better' if lo>0 else 'inconclusive')
            rows.append({'horizon_min':h,'block_days':L,'A':a,'B':b,
                         'delta_RMSE':observed,'CI95_low':float(lo),'CI95_high':float(hi),
                         'CI_excludes_zero':status!='inconclusive',
                         'conclusion':status,'bootstrap_fraction_A_better':float((diff<0).mean()),
                         'n_days':D,'n_nodes':13643,'n_replicates':B})
    print('Finished horizon:',h)

results=pd.DataFrame(rows)
assert len(results)==4*4*len(PAIRS)
assert results.groupby(['horizon_min','A','B']).delta_RMSE.nunique().eq(1).all()
baseline=pd.DataFrame(baseline_rows)
results.to_csv(OUT/'01_BLOCK_LENGTH_all_contrasts.csv',index=False)
baseline.to_csv(OUT/'02_model_metrics_recomputed.csv',index=False)

wide=results.pivot(index=['horizon_min','A','B'],columns='block_days',values='conclusion').reset_index()
wide.columns=[str(x) if isinstance(x,int) else x for x in wide.columns]
wide['stable_all_4']=wide[[str(x) for x in BLOCK_LENGTHS]].nunique(axis=1).eq(1)
wide['status_changes']=wide[[str(x) for x in BLOCK_LENGTHS]].nunique(axis=1)-1
wide.to_csv(OUT/'00_PRIMARY_sensitivity_summary.csv',index=False)
print(wide.to_string(index=False))
print('Stable contrasts:',int(wide.stable_all_4.sum()),'/',len(wide))


In [ ]:
# Contrôle : les points estimés sont identiques à ceux de E3-G, à l'arrondi près.
old=OUTPUTS/'E3_G_PAIRED_BLOCK_BOOTSTRAP'/'03_PAIRED_block_bootstrap_CI.csv'
if old.is_file():
    original=pd.read_csv(old)
    compare=results[results.block_days.eq(2)].merge(
        original[['horizon_min','A','B','delta_RMSE']],
        on=['horizon_min','A','B'],suffixes=('_G2','_G'))
    assert len(compare)==4*len(PAIRS)
    assert np.allclose(compare.delta_RMSE_G2,compare.delta_RMSE_G,atol=1e-9,rtol=0)
    compare.to_csv(OUT/'03_point_estimate_E3G_reconciliation.csv',index=False)
    print('E3-G point estimates reconciled:',len(compare))
else:
    print('Original E3-G CSV not found: point-estimate cross-check skipped.')

protocol={'experiment':'E3-G2 sensitivity to temporal bootstrap block length',
          'block_lengths_days':list(BLOCK_LENGTHS),'replications':B,
          'rng_seed_base':RNG_SEED,'horizons':list(HORIZONS),
          'contrasts':PAIRS,'sources':{k:str(v) for k,v in SOURCES.items()},
          'sampling_unit':'UTC day, all seven devices together',
          'seed_handling':'mean of per-seed RMSE, fixed 3 deep-model seeds',
          'CI':'percentile 95%, descriptive, no multiple-testing correction',
          'limitations':['22 UTC days only','3 training seeds',
                         '5-day blocks imply low effective temporal sample size',
                         'no generalization to other buildings',
                         'no isolated semantic-layer causal effect']}
(OUT/'04_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print('Saved:',OUT)


## Fichiers à transmettre

- `00_PRIMARY_sensitivity_summary.csv` — conclusions comparées pour 1, 2, 3 et 5 jours.
- `01_BLOCK_LENGTH_all_contrasts.csv` — différences et IC à 95 % (80 lignes).
- `02_model_metrics_recomputed.csv` — contrôle des RMSE.
- `03_point_estimate_E3G_reconciliation.csv` — contrôle facultatif avec E3-G.
- `04_protocol.json` — paramètres et limites.

**À interpréter avec prudence :** si les conclusions changent avec la longueur des blocs, indiquer explicitement cette sensibilité dans l'article. Si elles ne changent pas, la conclusion est robuste aux longueurs testées, mais pas nécessairement à une autre période de mesure.
